# Quickstart 3 &ndash; Investments & Storage

*Adapted from: [PyPSA Quickstart 3 &ndash; Investments & Storage](https://docs.pypsa.org/latest/examples/example-3/)*

In [ ]:
import math
from typing import cast

import pandas as pd
import pypsa

import typsa
from typsa.components import (
    Bus,
    Carrier,
    ExtendableGenerator,
    ExtendableStorageUnit,
    Generator,
    Load,
)
from typsa.time_variation import TimestampedSeries

## Defining the Network

In [2]:
def calculate_annuity(discount_rate: float, lifetime_years: int) -> float:
    return cast(float, pypsa.common.annuity(r=discount_rate, n=lifetime_years))

In [3]:
demand_p_set = 100
solar_p_max_pu = TimestampedSeries(
    pd.read_csv(
        "https://model.energy/data/time-series-f17c3736a2719ce7da58484180d89e2d.csv",
        index_col="time",
        parse_dates=["time"],
    )["solar"]
)
inverter_capital_cost_per_mw = 170_000 * calculate_annuity(
    discount_rate=0.05, lifetime_years=25
)
storage_capital_cost_per_mw = 150_000 * calculate_annuity(
    discount_rate=0.05, lifetime_years=25
)
battery_energy_to_power_ratio = 4
battery_round_trip_efficiency = 0.9

network = typsa.Network(snapshots=solar_p_max_pu.snapshots)

network.add_components(
    ac_carrier := Carrier(name="AC"),
    grid_carrier := Carrier(name="grid_carrier"),
    solar_carrier := Carrier(name="solar_carrier"),
    battery_carrier := Carrier(name="battery_carrier"),
)
network.add_components(
    seville_bus := Bus(name="seville"),
    demand := Load(
        name="demand", bus=seville_bus.name, carrier=ac_carrier.name, p_set=demand_p_set
    ),
    grid := Generator(
        name="grid",
        bus=seville_bus.name,
        p_nom=demand_p_set,
        marginal_cost=120,
        carrier=grid_carrier.name,
    ),
    solar := ExtendableGenerator(
        name="solar",
        bus=seville_bus.name,
        p_max_pu=solar_p_max_pu,
        capital_cost=(
            400_000 * calculate_annuity(discount_rate=0.05, lifetime_years=25)
        ),
        carrier=solar_carrier.name,
    ),
    battery := ExtendableStorageUnit(
        name="battery",
        bus=seville_bus.name,
        capital_cost=(
            inverter_capital_cost_per_mw
            + battery_energy_to_power_ratio * storage_capital_cost_per_mw
        ),
        carrier=battery_carrier.name,
        efficiency_store=math.sqrt(battery_round_trip_efficiency),
        efficiency_dispatch=math.sqrt(battery_round_trip_efficiency),
        max_hours=battery_energy_to_power_ratio,
    ),
)

## Optimizing the Network

In [4]:
optimized_network = network.model().optimize()

INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.: 100%|██████████| 6/6 [00:00<00:00, 840.35it/s]
INFO:linopy.io: Writing time: 0.13s


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-0veki95h has 105122 rows; 43802 cols; 188299 nonzeros
Coefficient ranges:
  Matrix  [1e-03, 4e+00]
  Cost    [1e+02, 5e+04]
  Bound   [0e+00, 0e+00]
  RHS     [1e+02, 1e+02]
Presolving model
48138 rows, 39380 cols, 126893 nonzeros  0s
41005 rows, 32250 cols, 118948 nonzeros  0s
Dependent equations search running on 9293 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
39883 rows, 31134 cols, 123758 nonzeros  0s
Presolve reductions: rows 39883(-65239); columns 31134(-12668); nonzeros 123758(-64541) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     1.0800000000e+05 Pr: 12168(2.155e+06) 0s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 43802 primals, 105122 duals
Objective: 4.83e+07
Solver model: available
Solver message: Optimal

INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, Generator-ext-p-lower, Generator-ext-p-upper, StorageUnit-ext-p_dispatch-lower, StorageUnit-ext-p_dispatch-upper, StorageUnit-ext-p_store-lower, StorageUnit-ext-p_store-upper, StorageUnit-ext-state_of_charge-lower, StorageUnit-ext-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


      22024     4.8284276524e+07 Pr: 0(0); Du: 0(4.15374e-11) 1s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name          : linopy-problem-0veki95h
Model status        : Optimal
Simplex   iterations: 22024
Objective value     :  4.8284276524e+07
P-D objective error :  1.5430655808e-16
HiGHS run time      :          1.53


In [5]:
optimized_network.static_results.all_capacities

{'solar': PNomOpt(value=659.7784140420765),
 'battery': PNomOpt(value=351.37183897980725)}